# 第5章：循环神经网络 (RNN)

> "RNN的核心：让神经网络拥有记忆。前面的词影响后面的理解——'我吃苹果'和'我买苹果手机'中的'苹果'完全不同。"

## 本章知识导图

```
循环神经网络 (Recurrent Neural Network)
│
├── 5.1 为什么需要RNN？全连接/CNN假设输入独立
├── 5.2 什么是RNN？
│   └── 核心公式: h_t = f(h_{t-1}, x_t)
│       隐藏状态在当前时刻和下一时刻间传递 → 这就是"记忆"
├── 5.3 RNN架构
│   ├── Elman网络：隐藏层自循环
│   └── Jordan网络：输出层自循环
├── 5.4 LSTM (长短期记忆网络)
│   ├── 细胞状态(Cell State)：穿越时间的"信息高速公路"
│   ├── 遗忘门 → 输入门 → 输出门
│   └── 为什么LSTM能解决梯度消失？加性更新！
├── 5.5 双向RNN：同时利用上下文信息
├── 5.6 梯度消失/爆炸的根源与解决方案
└── 5.7 RNN应用：多对一、多对多、Seq2Seq
```

## 5.0 序列建模：问题的本质

### 什么构成了序列？

序列数据在现实中无处不在：
- **文本**：词序列。"我/爱/吃/苹果" — 顺序至关重要
- **语音**：时间帧序列。每个时刻的频谱
- **视频**：帧序列。每帧依赖前几帧的动作
- **时间序列**：股票价格、气温、传感器读数
- **DNA**：碱基对序列

### 序列建模的核心挑战

1. **变长**：序列长度各异（一句话10个词，另一句100个词）
2. **长程依赖**：第100个词可能依赖第1个词（代词-先行词关系）
3. **顺序敏感**：交换两个词可能完全改变含义
4. **上下文依赖**：同一个词在不同上下文中含义不同

### 序列建模的三种范式

| 范式 | 输入→输出 | 示例 |
|------|----------|------|
| 多对一 (many-to-one) | 序列 → 单一输出 | 情感分析："这个电影很无聊" → 负面 |
| 一对一 (one-to-one) | 序列 → 等长序列 | 词性标注："I/love/AI" → "代词/动词/名词" |
| 多对多 (many-to-many) | 序列 → 不等长序列 | 机器翻译："I love AI" → "我 爱 人工智能" |

> RNN的设计动机正是处理这些变长序列——全连接和CNN要求固定大小输入，无法自然处理序列。</cell>


## 5.1 为什么需要RNN？

### 全连接和CNN的盲区
全连接和CNN都假设输入之间是**独立的**——每个样本独立处理。但语言、语音、视频都是**序列**，前面的信息影响后面的理解。

例子：
- "我吃了一个**苹果**" → 苹果是水果
- "我买了一个**苹果**手机" → 苹果是品牌
同一个词，不同上下文，含义不同。CNN做不到这种理解。

### RNN的核心思想：记忆
$$h_t = f(h_{t-1}, x_t; \theta)$$
当前时刻的隐藏状态$h_t$不仅取决于当前输入$x_t$，还取决于上一时刻的隐藏状态$h_{t-1}$。
$h_t$中保存了从序列开始到当前位置的"记忆"。

> **关键：** 所有时间步**共享同一组参数**$\theta$——RNN的参数量不随序列长度增长，无论处理10个词还是1000个词。

## 5.2.1 BPTT：时间反向传播详解

### BPTT的核心思想

RNN在时间维度上展开后，本质上是一个**参数共享的深层网络**。BPTT (Backpropagation Through Time) 就是对这个展开后的网络做反向传播。

### 具体展开（以3个时间步为例）

前向传播：
```
t=1:  h_1 = tanh(W_h·h_0 + W_x·x_1 + b)    →  y_1 = W_y·h_1 + b_y
t=2:  h_2 = tanh(W_h·h_1 + W_x·x_2 + b)    →  y_2 = W_y·h_2 + b_y
t=3:  h_3 = tanh(W_h·h_2 + W_x·x_3 + b)    →  y_3 = W_y·h_3 + b_y
```

沿时间维度展开的可视化：

```
      L_1         L_2         L_3
       ↑           ↑           ↑
      y_1         y_2         y_3
       ↑           ↑           ↑
h_0 → [W_h] → h_1 → [W_h] → h_2 → [W_h] → h_3
       ↑           ↑           ↑
      x_1         x_2         x_3
```

注意：每个箭头处的 W_h 是**同一组参数**！

### BPTT的梯度流

在t=3时刻的损失$L_3$对W_h的梯度，需要沿时间回传：

$$\frac{\partial L_3}{\partial W_h} = \frac{\partial L_3}{\partial h_3} \cdot \frac{\partial h_3}{\partial W_h} + \frac{\partial L_3}{\partial h_3} \cdot \frac{\partial h_3}{\partial h_2} \cdot \frac{\partial h_2}{\partial W_h} + \frac{\partial L_3}{\partial h_3} \cdot \frac{\partial h_3}{\partial h_2} \cdot \frac{\partial h_2}{\partial h_1} \cdot \frac{\partial h_1}{\partial W_h}$$

关键项：$\frac{\partial h_t}{\partial h_{t-1}}$ — 每往后回传一步就乘以一次这个雅可比矩阵。

### 梯度消失/爆炸的数学根源

对于标量简化（实际是矩阵，但原理相通）：

$$\frac{\partial L_T}{\partial h_1} = \frac{\partial L_T}{\partial h_T} \cdot \prod_{t=2}^{T} \frac{\partial h_t}{\partial h_{t-1}}$$

其中 $\frac{\partial h_t}{\partial h_{t-1}} = W_h \cdot \text{diag}(\tanh'(W_h h_{t-1} + W_x x_t))$

- $\tanh'$ 的最大值是1（在x=0处），通常在 (0, 1) 范围内
- 如果$|W_h| < 1$ → 连乘T次 → 趋近于0 → **梯度消失**
- 如果$|W_h| > 1$ → 连乘T次 → 趋近于∞ → **梯度爆炸**

### 具体数值演示

```python
# 梯度消失的数学演示
w = 0.5  # |W_h| < 1
T = 100  # 序列长度
factor = w ** T
print(f"衰减因子: {factor:.2e}")  # 约 7.9e-31 → 几乎为0！

# 梯度爆炸的数学演示
w = 1.5  # |W_h| > 1
factor = w ** T
print(f"放大因子: {factor:.2e}")  # 约 4.1e+17 → 爆炸！
```

> **BPTT的关键洞察**：RNN的误差信号需要穿越所有时间步才能到达远处的参数。每穿越一步就乘以一次"衰减因子"。当序列很长时，远处的误差信号几乎消失（梯度消失）或失控（梯度爆炸）。这导致RNN"记不住"远距离的信息。</cell>


## 5.3.2 GRU：LSTM的简化版

### GRU (Gated Recurrent Unit) — 2014年提出

GRU是LSTM的简化版本，将三个门精简为两个门，去除了独立的细胞状态。

### GRU的核心公式

$$\begin{align}
z_t &= \sigma(W_z · [h_{t-1}, x_t] + b_z) \quad &\text{更新门 (Update Gate)}\\
r_t &= \sigma(W_r · [h_{t-1}, x_t] + b_r) \quad &\text{重置门 (Reset Gate)}\\
\tilde{h}_t &= \tanh(W_h · [r_t \odot h_{t-1}, x_t] + b_h) \quad &\text{候选隐藏状态}\\
h_t &= (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t \quad &\text{最终隐藏状态}
\end{align}$$

### GRU vs LSTM的关键差异

| 对比项 | LSTM | GRU |
|--------|------|-----|
| 门数量 | 3个 (遗忘、输入、输出) | 2个 (重置、更新) |
| 状态数量 | 2个 ($c_t$, $h_t$) | 1个 ($h_t$) |
| 梯度流动 | 通过$c_t$的加法 | 通过$h_t$的插值 |
| 参数量 | 4×d×2d | 3×d×2d |
| 训练速度 | 较慢 | 较快 (~少25%参数) |
| 性能 | 理论更灵活 | 实践中相近 |

### 门的作用

- **重置门 $r_t$**：决定如何将新输入与之前的记忆结合。$r_t \approx 0$ → 忽略历史，像在读新序列的第一个词
- **更新门 $z_t$**：决定保留多少旧状态和多少新候选。$z_t \approx 1$ → 几乎完全复制旧状态，信息长期保留；$z_t \approx 0$ → 几乎完全替换为新信息

### 关键洞察：GRU的插值机制

$$h_t = (1-z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

这是$h_{t-1}$和$\tilde{h}_t$之间的**线性插值**(convex combination)！因为$z_t \in (0,1)$，两者的权重互补（和为1）。

- 当$z_t$接近0：$h_t \approx h_{t-1}$（保留旧记忆）
- 当$z_t$接近1：$h_t \approx \tilde{h}_t$（用新信息覆盖）
- 当$z_t=0.5$：新旧信息各占一半

> **实践建议：** 没有明确的"LSTM一定优于GRU"或反之。一般来说，数据量不大时GRU可能更好（参数少，不易过拟合），大数据量时LSTM的额外表达能力可能有用。两者都被现代Transformer所取代。</cell>


In [ ]:
import torch
import torch.nn as nn
import numpy as np

# ============================================================
# 字符级文本生成 LSTM — 学习预测下一个字符
# ============================================================
print("=" * 60)
print("字符级LSTM文本生成")
print("=" * 60)

# 1. 准备数据：使用一段中文文本
text = ("深度学习是人工智能的一个重要分支。"
        "卷积神经网络擅长处理图像，"
        "循环神经网络擅长处理序列数据，"
        "而Transformer模型则在自然语言处理中取得了革命性突破。"
        "深度学习已经广泛应用于计算机视觉、语音识别、自然语言处理等领域。")

chars = sorted(list(set(text)))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}
vocab_size = len(chars)
print(f"词汇表大小: {vocab_size}")
print(f"字符: {''.join(chars)}")

# 2. 创建训练样本：用前seq_len个字符预测下一个
seq_len = 20
X_data, y_data = [], []
for i in range(0, len(text) - seq_len):
    X_data.append([char_to_idx[c] for c in text[i:i+seq_len]])
    y_data.append(char_to_idx[text[i+seq_len]])

X_data = torch.tensor(X_data)
y_data = torch.tensor(y_data)
print(f"训练样本数: {len(X_data)}")
print(f"X shape: {X_data.shape}, y shape: {y_data.shape}")

# 3. 定义LSTM模型
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, 
                            batch_first=True, dropout=0.3)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x, hidden=None):
        # x: (batch, seq_len)
        x = self.embedding(x)                    # (B, L, E)
        out, hidden = self.lstm(x, hidden)       # out: (B, L, H)
        out = self.fc(out[:, -1, :])             # 只取最后一个时间步, (B, V)
        return out, hidden

# 4. 训练
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CharLSTM(vocab_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

batch_size = 16
num_epochs = 100

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    # 随机抽样batch
    perm = torch.randperm(len(X_data))
    for i in range(0, len(X_data), batch_size):
        idx = perm[i:i+batch_size]
        x_batch = X_data[idx].to(device)
        y_batch = y_data[idx].to(device)
        
        optimizer.zero_grad()
        outputs, _ = model(x_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(X_data)*batch_size:.4f}")

print("训练完成！\n")

# 5. 生成文本
print("=" * 60)
print("文本生成")
print("=" * 60)

def generate_text(model, start_str, length=100, temperature=0.8):
    model.eval()
    # 将种子文本转为索引
    input_seq = torch.tensor([[char_to_idx.get(c, 0) for c in start_str]]).to(device)
    hidden = None
    generated = start_str
    
    with torch.no_grad():
        for _ in range(length):
            output, hidden = model(input_seq, hidden)
            # 温度采样
            logits = output[0] / temperature
            probs = torch.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, 1).item()
            next_char = idx_to_char[next_idx]
            generated += next_char
            # 更新输入（只用最新字符，让hidden保持上下文）
            input_seq = torch.tensor([[next_idx]]).to(device)
    
    return generated

seed = "深度学习"
generated = generate_text(model, seed, length=80, temperature=0.8)
print(f"种子: {seed}")
print(f"生成: {generated}")

# 不同温度的效果
print("\n=== 温度参数的影响 ===")
print("Temperature=0.3 (保守):")
print(generate_text(model, seed, length=50, temperature=0.3))
print("\nTemperature=1.5 (创造性):")
print(generate_text(model, seed, length=50, temperature=1.5))</cell>


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# ============================================================
# Seq2Seq LSTM：编码器-解码器架构
# ============================================================
print("=" * 60)
print("Seq2Seq 编码器-解码器 LSTM")
print("=" * 60)

class EncoderLSTM(nn.Module):
    """编码器：读取整个输入序列，输出上下文向量"""
    def __init__(self, input_vocab_size, embed_dim, hidden_dim, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(input_vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True)
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
    
    def forward(self, x):
        # x: (batch, src_seq_len)
        embedded = self.embedding(x)                    # (B, S, E)
        outputs, (hidden, cell) = self.lstm(embedded)   # outputs: (B, S, H)
        # 返回所有时间步的输出 + 最后的隐藏/细胞状态（上下文向量）
        return outputs, (hidden, cell)

class DecoderLSTM(nn.Module):
    """解码器：基于编码器的上下文向量，自回归生成输出序列"""
    def __init__(self, output_vocab_size, embed_dim, hidden_dim, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(output_vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, output_vocab_size)
    
    def forward(self, x, hidden_cell):
        # x: (batch, tgt_seq_len) — 训练时一次输入整个目标序列（Teacher Forcing）
        embedded = self.embedding(x)                    # (B, T, E)
        outputs, (hidden, cell) = self.lstm(embedded, hidden_cell)
        predictions = self.fc_out(outputs)              # (B, T, V_out)
        return predictions, (hidden, cell)

class Seq2Seq(nn.Module):
    """完整的序列到序列模型"""
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
    
    def forward(self, src, tgt):
        # src: 源序列, tgt: 目标序列（训练用Teacher Forcing）
        _, context = self.encoder(src)
        output, _ = self.decoder(tgt, context)
        return output

# 创建dummy数据演示
src_vocab_size = 100   # 源语言词汇量
tgt_vocab_size = 100   # 目标语言词汇量
embed_dim = 64
hidden_dim = 128

encoder = EncoderLSTM(src_vocab_size, embed_dim, hidden_dim)
decoder = DecoderLSTM(tgt_vocab_size, embed_dim, hidden_dim)
model = Seq2Seq(encoder, decoder)

# 模拟数据：batch=4, 源序列长度=7, 目标序列长度=5
src = torch.randint(0, src_vocab_size, (4, 7))  # 源语言句子（7个词）
tgt = torch.randint(0, tgt_vocab_size, (4, 5))  # 目标语言句子（5个词）

output = model(src, tgt)
print(f"源输入: {src.shape}")
print(f"目标输入: {tgt.shape}")
print(f"模型输出: {output.shape}  ← (batch, tgt_len, vocab_size)")
print(f"每个位置输出vocab_size个logits → 取argmax得到预测token")

# 演示推理时的自回归生成
print(f"\n=== 推理时的自回归生成 ===")
print("训练时: 一次性输入整个目标序列(Teacher Forcing)")
print("推理时: 逐token生成:")
print("  1. 输入<SOS> → 输出'我'")
print("  2. 输入<SOS>+'我' → 输出'爱'")
print("  3. 输入<SOS>+'我'+'爱' → 输出'AI'")
print("  4. ...直到输出<EOS>或达到最大长度")
print("\n关键：编码器只运行一次（得到上下文向量），解码器循环运行N次（N=生成token数）")</cell>


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 时间序列预测：用LSTM预测正弦波
# ============================================================
print("=" * 60)
print("LSTM时间序列预测")
print("=" * 60)

# 1. 生成正弦波数据
def generate_sine_data(seq_length=1000, period=50):
    t = np.arange(seq_length)
    # 复合正弦波（多个频率叠加）
    data = np.sin(2 * np.pi * t / period) + 0.3 * np.sin(2 * np.pi * t / (period * 2.5))
    data = data.astype(np.float32)
    return data, t

data, t = generate_sine_data(seq_length=800)

# 可视化原始数据
plt.figure(figsize=(14, 3))
plt.plot(t[:200], data[:200], label='Sine Wave')
plt.title('时间序列数据（前200个点）')
plt.legend()
plt.show()

# 2. 准备训练数据：用前lookback个点预测下一个点
def create_sequences(data, lookback=50):
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:i+lookback])
        y.append(data[i+lookback])
    return torch.tensor(np.array(X)).unsqueeze(-1), torch.tensor(np.array(y)).unsqueeze(-1)

lookback = 50
X, y = create_sequences(data, lookback)
print(f"X shape: {X.shape}  (样本数, 回看长度, 特征数)")
print(f"y shape: {y.shape}  (样本数, 1)")

# 划分训练/测试
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
print(f"训练: {X_train.shape[0]} 样本, 测试: {X_test.shape[0]} 样本")

# 3. LSTM预测模型
class TimeSeriesLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, 
                            batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        out, _ = self.lstm(x)          # (B, L, H)
        out = self.fc(out[:, -1, :])   # 只取最后一步, (B, 1)
        return out

# 4. 训练
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ts_model = TimeSeriesLSTM().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(ts_model.parameters(), lr=0.001)

X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

num_epochs = 50
for epoch in range(num_epochs):
    ts_model.train()
    optimizer.zero_grad()
    pred = ts_model(X_train)
    loss = criterion(pred, y_train)
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        ts_model.eval()
        with torch.no_grad():
            test_pred = ts_model(X_test)
            test_loss = criterion(test_pred, y_test)
        print(f"Epoch [{epoch+1}/{num_epochs}] 训练Loss: {loss.item():.5f}  测试Loss: {test_loss.item():.5f}")

# 5. 预测与可视化
ts_model.eval()
with torch.no_grad():
    train_pred = ts_model(X_train).cpu().numpy()
    test_pred = ts_model(X_test).cpu().numpy()

plt.figure(figsize=(14, 5))
# 绘制全部数据
plt.plot(range(len(data)), data, 'b-', alpha=0.3, label='实际数据')
# 训练集预测
train_range = range(lookback, lookback + len(train_pred))
plt.plot(train_range, train_pred, 'g-', linewidth=1, label='训练集预测')
# 测试集预测
test_range = range(lookback + len(train_pred), lookback + len(train_pred) + len(test_pred))
plt.plot(test_range, test_pred, 'r-', linewidth=1.5, label='测试集预测')
plt.axvline(x=lookback + len(train_pred), color='k', linestyle='--', alpha=0.5, label='训练/测试分割')
plt.title('LSTM时间序列预测 — 正弦波')
plt.legend()
plt.show()

# 6. 多步预测（用自己的预测继续预测）
print(f"\n=== 多步预测（自回归预测未来） ===")
ts_model.eval()
with torch.no_grad():
    # 取测试集最后一个序列作为起点
    last_seq = X_test[-1:]  # (1, lookback, 1)
    multi_preds = []
    current = last_seq.clone()
    for _ in range(100):  # 预测未来100步
        pred = ts_model(current).cpu().item()
        multi_preds.append(pred)
        # 滑动窗口：去掉最早的值，加入预测值
        current = torch.cat([current[:, 1:, :], 
                             torch.tensor([[[pred]]]).to(device)], dim=1)

plt.figure(figsize=(14, 5))
plt.plot(range(len(data)), data, 'b-', alpha=0.5, label='历史数据')
future_range = range(len(data), len(data) + 100)
plt.plot(future_range, multi_preds, 'r-', linewidth=2, label='未来预测（100步）')
plt.axvline(x=len(data), color='k', linestyle='--', alpha=0.5)
plt.title('多步预测：用LSTM预测不可见的未来')
plt.legend()
plt.show()

print("\n=== 时间序列预测要点 ===")
print("1. 滑动窗口 = 将时间序列转为有监督学习（前N个 → 下一个）")
print("2. LSTM自动学习时间依赖模式（周期性、趋势等）")
print("3. 多步预测存在误差累积问题（预测误差反馈到下一轮输入）")
print("4. 改进方向：添加注意力机制、用Transformer替代LSTM、多变量输入预测")</cell>


## 5.6 RNN的应用全景与局限性

### RNN的应用场景汇总

| 应用领域 | 具体任务 | RNN类型 | 输入→输出 |
|----------|----------|---------|-----------|
| NLP | 情感分析 | 多对一 | 句子→情感标签 |
| NLP | 词性标注(POS Tagging) | 一对一 | 每个词→词性标签 |
| NLP | 命名实体识别(NER) | 一对一 | 每个词→BIO标签 |
| NLP | 机器翻译 | 多对多(Seq2Seq) | 源语言→目标语言 |
| NLP | 文本生成 | 多对多(自回归) | 前文→后续文本 |
| 语音 | 语音识别(ASR) | 多对多(Seq2Seq) | 声学特征→文本 |
| 语音 | 语音合成(TTS) | 多对多 | 文本→声学特征 |
| 视频 | 动作识别 | 多对一 | 帧序列→动作标签 |
| 视频 | 视频描述(Video Captioning) | 多对多 | 帧序列→文字描述 |
| 时序 | 股票预测 | 多对一/多对多 | 历史价格→未来价格 |
| 生物 | 蛋白质结构预测 | 多对一 | 氨基酸序列→结构 |
| 生物 | DNA序列分析 | 多对多 | DNA序列→功能标注 |

### RNN的致命弱点（为什么后来被Transformer取代）

1. **串行依赖（无法并行）**：计算$h_t$必须先计算$h_{t-1}$。这使得RNN无法利用GPU的大规模并行能力。序列长度1000 = 必须做1000次串行计算。

2. **长程依赖依然有限**：虽然LSTM/GRU改善了梯度消失，但当序列长度超过几百时，长程依赖仍然不可靠。信息每过一个时间步就有一定程度的衰减。

3. **训练慢**：BPTT需要展开整个序列，内存和计算开销与序列长度成正比。

4. **Exposure Bias**：训练用Teacher Forcing（看正确答案），推理看自己生成的输出 → 错误会累积。

5. **不适合超长序列**：对于上万个token的文档，RNN的串行本质使其极其低效。

> **为什么第6、7章的Transformer取代了RNN？** 因为自注意力机制让每个位置可以直接访问所有其他位置，完全不需要串行传播。而且所有位置的计算可以同时进行（并行），训练速度成倍提升。这些优势使Transformer成为NLP乃至更多领域的标准架构。</cell>


## 5.7 本章知识总结与思考题

### 核心公式汇总

| 模型 | 核心更新公式 | 梯度优势 |
|------|-------------|----------|
| 标准RNN | $h_t = \tanh(W_h h_{t-1} + W_x x_t)$ | 无（乘法链） |
| LSTM | $c_t = f_t \odot c_{t-1} + i_t \odot g_t$ | **加法**操作 → 梯度无衰减 |
| GRU | $h_t = (1-z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$ | **插值** → 线性组合路径 |

### 核心知识回顾

1. **RNN本质**：通过$h_t$在时间步间传递信息，实现序列"记忆"
2. **参数共享**：所有时间步共享同一组权重 — 参数量不随序列长度增长
3. **BPTT**：沿时间展开的反向传播，梯度连乘导致消失/爆炸
4. **LSTM三把钥匙**：遗忘门(忘记旧的) + 输入门(写入新的) + 输出门(输出选择的)
5. **加法是LSTM的精髓**：$c_t = f \odot c_{t-1} + i \odot g$中的"+"让梯度无衰减穿越时间
6. **GRU = 简化版LSTM**：合并遗忘/输入门为更新门，去除独立细胞状态
7. **Seq2Seq**：编码器压缩输入→上下文，解码器自回归生成输出
8. **RNN的核心局限**：串行处理无法并行 → 被Transformer（第6-7章）取代

### 思考题

1. 为什么LSTM的加法操作能解决梯度消失，而标准RNN的乘法操作不行？从$\frac{\partial c_t}{\partial c_{t-1}}$的角度解释。
2. 双向LSTM在命名实体识别中为什么特别有效？它在预测第i个词的标签时能看到什么信息？
3. Teacher Forcing训练和自回归推理之间的"暴露偏差"(Exposure Bias)问题如何缓解？
4. 如果LSTM的遗忘门全部输出1（永远不遗忘），输入门全部输出0（永不写入），会发生什么？
5. 为什么字符级文本生成比词级更灵活但也有更多挑战？

### 延伸阅读
- "Long Short-Term Memory" (Hochreiter & Schmidhuber, 1997) — LSTM原始论文
- "Learning Phrase Representations using RNN Encoder-Decoder" (Cho et al., 2014) — GRU论文
- "Sequence to Sequence Learning with Neural Networks" (Sutskever et al., 2014) — Seq2Seq
- "The Unreasonable Effectiveness of Recurrent Neural Networks" (Karpathy, 2015) — 经典博客</cell>


In [ ]:
print(f"  GRU单元:  {gru_params}  (= 3 × (input+h) × h)")

## 5.3.1 LSTM完整走查：一个玩具示例

### 用具体数值演示LSTM的前向传播

假设我们有一个简化的LSTM（hidden_size=2），处理两个时间步的输入。我们将演示每个门如何通过具体的数值完成信息的选择性保留/遗忘。

**模型设定（简化值，仅用于演示）：**
- 输入维度: 1
- 隐藏维度: 2
- 初始状态: $h_0 = [0, 0]$, $c_0 = [0, 0]$

**时间步 t=1:**
- 输入: $x_1 = 1.0$

**Step 1 — 遗忘门 $f_1$：** 决定丢弃多少旧信息
$$f_1 = \sigma(W_f · [h_0, x_1] + b_f)$$

假设简化后：$f_1 = \sigma([0.3, -0.2]) = [0.574, 0.450]$
含义：保留57%和45%的旧细胞状态。由于$c_0=[0,0]$，这个门暂时没有什么可遗忘的。

**Step 2 — 输入门 $i_1$ 和候选值 $g_1$：** 决定写入什么新信息
$$i_1 = \sigma(W_i · [h_0, x_1] + b_i)$$
$$g_1 = \tanh(W_g · [h_0, x_1] + b_g)$$

假设：$i_1 = \sigma([2.0, 1.5]) = [0.881, 0.818]$
      $g_1 = \tanh([1.0, -0.5]) = [0.762, -0.462]$

**Step 3 — 更新细胞状态：** 核心！加法操作
$$c_1 = f_1 \odot c_0 + i_1 \odot g_1$$
$$c_1 = [0.574 \times 0, 0.450 \times 0] + [0.881 \times 0.762, 0.818 \times (-0.462)]$$
$$c_1 = [0, 0] + [0.671, -0.378] = [0.671, -0.378]$$

**Step 4 — 输出门 $o_1$：** 决定输出多少
$$o_1 = \sigma(W_o · [h_0, x_1] + b_o) = \sigma([0.5, -0.1]) = [0.622, 0.475]$$
$$h_1 = o_1 \odot \tanh(c_1) = [0.622 \times \tanh(0.671), 0.475 \times \tanh(-0.378)]$$
$$h_1 = [0.622 \times 0.585, 0.475 \times (-0.361)] = [0.364, -0.171]$$

**时间步 t=2:**
- 输入: $x_2 = -0.5$
- 现在我们有了非零的$h_1$和$c_1$

遗忘门看到$h_1=[0.364, -0.171]$和$x_2=-0.5$：
$$f_2 = \sigma([1.5, -1.0]) = [0.818, 0.269]$$

遗忘门告诉LSTM："丢弃$c_1$中'不相关'的部分"。$c_1=[0.671,-0.378]$中，第二个维度将被遗忘73%！

输入门和候选：
$$i_2 = \sigma([0.3, 2.0]) = [0.574, 0.881]$$
$$g_2 = \tanh([-0.2, 1.5]) = [-0.197, 0.905]$$

更新：
$$c_2 = [0.818,0.269] \odot [0.671,-0.378] + [0.574,0.881] \odot [-0.197, 0.905]$$
$$= [0.549, -0.102] + [-0.113, 0.797] = [0.436, 0.695]$$

输出：
$$h_2 = o_2 \odot \tanh([0.436, 0.695]) = [0.254, 0.471]$$

### 关键观察

1. **$c_t$通过加法更新**：$c_2$来自两部分——$f \odot c_1$（保留的旧信息）+ $i \odot g$（新写入的信息）。**加法**意味着梯度可以无衰减地通过！
2. **门是"软"开关**：遗忘门不是完全丢弃（0或1），而是0~1之间的连续值 → 可微分
3. **不同维度独立决策**：$c_t$的每个维度有自己独立的门 → 可以同时记住不同类型的信息

> **直觉理解**：$c_t$像一张"草稿纸"，遗忘门擦除不需要的内容，输入门写入新内容，输出门决定哪些内容要"公开"（输出到$h_t$）。这三个门的配合使LSTM能选择性保留远距离信息。</cell>


In [ ]:
import torch
import torch.nn as nn

# ============================================================
# 梯度裁剪演示 — 防止RNN训练时的梯度爆炸
# ============================================================
print("=" * 60)
print("梯度裁剪(Gradient Clipping)演示")
print("=" * 60)

# 创建一个简单的RNN并模拟梯度爆炸
rnn = nn.RNN(input_size=10, hidden_size=20, num_layers=3)
x = torch.randn(5, 100, 10)  # (seq_len=100, batch=5, feature=10)
output, hidden = rnn(x)

# 模拟一个大梯度（通常由长序列训练引发）
loss = output.sum()
loss.backward()

# 查看梯度范数
total_norm = 0
for p in rnn.parameters():
    if p.grad is not None:
        param_norm = p.grad.data.norm(2)
        total_norm += param_norm.item() ** 2
total_norm = total_norm ** 0.5
print(f"裁剪前梯度总范数: {total_norm:.2f}")

# 执行梯度裁剪
max_norm = 1.0
torch.nn.utils.clip_grad_norm_(rnn.parameters(), max_norm=max_norm)

# 查看裁剪后的梯度范数
total_norm_after = 0
for p in rnn.parameters():
    if p.grad is not None:
        param_norm = p.grad.data.norm(2)
        total_norm_after += param_norm.item() ** 2
total_norm_after = total_norm_after ** 0.5
print(f"裁剪后梯度总范数: {total_norm_after:.2f}")
print(f"裁剪比例: {total_norm_after/total_norm:.3f}")

print(f"\n=== 梯度裁剪原理 ===")
print("if ||g|| > max_norm: g = g * (max_norm / ||g||)")
print("简单来说：梯度太大就等比例缩小，保持方向不变")
print(f"\n=== 为什么RNN需要梯度裁剪而CNN不需要？ ===")
print("RNN: BPTT梯度连乘 → 长序列时梯度可能指数增长 → 必须裁剪")
print("CNN: 通常较浅(< 100层)，且BatchNorm/ResNet已缓解梯度问题")
print("但深层Transformer也需要梯度裁剪（注意力矩阵的梯度范围很大）")</cell>


## 5.2 RNN的展开视角

RNN可以看作在时间维度上的"深层网络"：

```
h_0 → [RNN cell] → h_1 → [RNN cell] → h_2 → [RNN cell] → h_3 → ...
         ↑               ↑               ↑
        x_1             x_2             x_3
```

每个[RNN cell]用的是**同一组参数**！这和CNN的权重共享是同样的思想——只是共享的维度从"空间"变成了"时间"。

### Elman网络 vs Jordan网络
- **Elman网络**：$h_t$存储隐藏状态，下一时刻作为输入 → 更常用
- **Jordan网络**：存储输出$y_{t-1}$，下一时刻作为输入 → 理论上有优势（输出有明确语义），但实践中Elman更灵活

## 5.3 LSTM：解决长程依赖

### 传统RNN的致命弱点
当序列很长时（比如100个词），RNN的**梯度会消失**：
- 反向传播时梯度在每个时间步连乘
- 如果连乘因子<1 → 指数衰减 → 第100个词的错误传不到第1个词
- 结果：RNN只能记住"最近"的信息，远处的信息被遗忘

### LSTM的关键创新：门控 + 加性更新

LSTM引入**细胞状态(Cell State)** $c_t$——一条穿越所有时间步的"信息高速公路"：

$$c_t = f_t \odot c_{t-1} + i_t \odot g_t$$

这是**加法**操作！不像传统RNN的乘法，加法的梯度流不会指数衰减。

三个门：
- **遗忘门$f_t$**：$\sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$ → 控制丢弃多少旧信息(0~1)
- **输入门$i_t$**：$\sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$ → 控制存入多少新信息(0~1)
- **输出门$o_t$**：$\sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$ → 控制输出多少(0~1)

$$h_t = o_t \odot \tanh(c_t)$$

> **核心洞察：** $c_t = f_t \odot c_{t-1} + i_t \odot g_t$中的加号(+)是LSTM精髓——反向传播时梯度通过加法路径无衰减地穿越时间，解决了传统RNN的梯度消失。

In [ ]:
import torch
import torch.nn as nn

# PyTorch LSTM 详解
input_size = 10   # 输入特征维度（如词向量维度）
hidden_size = 20  # 隐藏状态/细胞状态的维度
num_layers = 2    # 堆叠层数

lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)

# 输入: batch=3, seq_len=5, feature_dim=10
x = torch.randn(3, 5, 10)
output, (h_n, c_n) = lstm(x)

print(f"输入shape: {x.shape}      (batch=3, seq_len=5, feature=10)")
print(f"输出shape: {output.shape}  (batch, seq_len, hidden_size)")
print(f"h_n shape: {h_n.shape}     (num_layers, batch, hidden_size)")
print(f"c_n shape: {c_n.shape}     (num_layers, batch, hidden_size)")
print()
print("output: 每个时间步的最后一个隐藏层输出")
print("h_n:    每个层的最后一个时间步的隐藏状态")
print("c_n:    每个层的最后一个时间步的细胞状态")

# 情感分类示例
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, num_layers=2, dropout=0.3)
        self.fc = nn.Linear(hidden_dim, num_classes)
    def forward(self, x):
        embedded = self.embedding(x)          # (B, L) → (B, L, E)
        _, (h_n, _) = self.lstm(embedded)     # 取最后一层最后时刻的隐藏状态
        return self.fc(h_n[-1])               # h_n[-1] = 最后一层

model = SentimentLSTM(vocab_size=5000, embed_dim=100, hidden_dim=64, num_classes=2)
input_ids = torch.randint(0, 5000, (16, 30))  # 16条评论，每条30个词
print(f"\n情感分类: input {input_ids.shape} → output {model(input_ids).shape}")

## 5.4 双向RNN

标准RNN只看"历史"（从左到右）。但有时"未来"信息也很重要：
- "我今天_**很**_开心" → 看到后面的"开心"有助于理解中间的情感
- 命名实体识别："苹果公司发布了新手机" → "苹果"后面的"公司"提示这不是水果

双向RNN：同时跑一个**正向**和一个**反向**RNN，每个时间步将两个方向的隐藏状态拼接起来。

> 注意：双向RNN不适合**实时**任务（需要看到"未来"才能输出）。

## 5.5 梯度消失/爆炸的解决方案

| 问题 | 原因 | 解法 |
|------|------|------|
| 梯度消失 | BPTT梯度连乘<1 → 指数衰减 | LSTM/GRU的门控机制 |
| 梯度爆炸 | BPTT梯度连乘>1 → 指数爆炸 | 梯度裁剪(Gradient Clipping) |

```python
# 梯度裁剪
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
```

## 本章核心收获
1. RNN通过隐藏状态$h_t$在时间步间传递信息，实现"记忆"
2. LSTM用三个门+加性更新$c_t$解决梯度消失
3. BPTT(时间反向传播)展开RNN，但梯度连乘导致消失/爆炸
4. 梯度裁剪防止爆炸，LSTM/GRU缓解消失
5. 双向RNN利用上下文，Seq2Seq实现序列转换
6. RNN的串行本质使其难以并行 → 第6章自注意力机制取而代之